# Logistic Regression

**Goal:** Implement binary logistic regression from scratch in PyTorch using sigmoid + BCE loss + gradient descent, validate against scikit-learn, and visualize the learned decision boundary.

## Configuration

In [1]:
import sys
from pathlib import Path

import matplotlib

matplotlib.use("Agg")  # headless-safe under nbconvert
import matplotlib.pyplot as plt  # noqa: E402
import torch  # noqa: E402


def _find_repo_root(start: Path) -> Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    return start


REPO_ROOT = _find_repo_root(Path.cwd())
sys.path.insert(0, str(REPO_ROOT))

from shared.config import configure  # noqa: E402

device = configure()
print("running on:", device)

running on: mps


## Imports and Synthetic Dataset

In [2]:
import numpy as np
import torch
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

torch.manual_seed(42)
np.random.seed(42)

n = 400  # total samples (balanced classes)

# Two linearly-separable-ish clusters in 2D
X0 = torch.randn(n // 2, 2) + torch.tensor([-1.5, -1.5])  # class 0
X1 = torch.randn(n // 2, 2) + torch.tensor([ 1.5,  1.5])  # class 1

X_cpu = torch.cat([X0, X1], dim=0)  # (n, 2)
y_cpu = torch.cat([torch.zeros(n // 2), torch.ones(n // 2)])  # (n,)

# Shuffle
idx = torch.randperm(n)
X_cpu = X_cpu[idx]
y_cpu = y_cpu[idx]

print(f"Dataset: X shape={X_cpu.shape}, y shape={y_cpu.shape}")
print(f"Class balance: {y_cpu.mean():.3f} positive rate")

Dataset: X shape=torch.Size([400, 2]), y shape=torch.Size([400])
Class balance: 0.500 positive rate


## From Scratch: Sigmoid, BCE, and Gradient Descent

### Key math

Given feature matrix `X` (n, 2) and bias `b`:

$$z_i = x_i^\top w + b, \quad p_i = \sigma(z_i) = \frac{1}{1 + e^{-z_i}}$$

Binary cross-entropy loss (negative log-likelihood of a Bernoulli):

$$L(w, b) = -\frac{1}{n} \sum_i \bigl[ y_i \log p_i + (1 - y_i) \log(1 - p_i) \bigr]$$

The analytic gradients are elegantly simple:

$$\nabla_w L = \frac{1}{n} X^\top (p - y), \quad \nabla_b L = \frac{1}{n} \sum_i (p_i - y_i)$$

### Numerical stability note

We use `torch.nn.functional.binary_cross_entropy_with_logits` (BCE-with-logits) which computes `log(sigmoid(z))` as `log(1 + exp(-z))` internally, avoiding overflow for large positive `z` and underflow for large negative `z`.

In [3]:
import torch.nn.functional as F


def sigmoid(z: torch.Tensor) -> torch.Tensor:
    """Numerically stable sigmoid via torch."""
    return torch.sigmoid(z)


def logistic_grad_descent(
    X: torch.Tensor,
    y: torch.Tensor,
    lr: float = 0.5,
    n_iters: int = 2000,
) -> tuple:
    """Train binary logistic regression via batch gradient descent.

    Uses BCE-with-logits for numerical stability.

    Args:
        X: Feature matrix (n, d).
        y: Binary labels (n,) in {0, 1}.
        lr: Learning rate.
        n_iters: Number of gradient steps.

    Returns:
        Tuple (w, b, loss_history).
    """
    n_samples, d = X.shape
    w = torch.zeros(d, dtype=X.dtype)
    b = torch.zeros(1, dtype=X.dtype)
    losses = []

    for i in range(n_iters):
        z = X @ w + b                              # logits (n,)
        loss = F.binary_cross_entropy_with_logits(z, y)

        # Analytic gradients: grad_w = (1/n) X^T (sigmoid(z) - y)
        p = sigmoid(z)
        err = p - y                                # (n,)
        grad_w = (X.T @ err) / n_samples          # (d,)
        grad_b = err.mean().unsqueeze(0)           # (1,)

        w = w - lr * grad_w
        b = b - lr * grad_b

        if i % 200 == 0:
            losses.append(loss.item())

    return w, b, losses


w_scratch, b_scratch, loss_history = logistic_grad_descent(
    X_cpu.float(), y_cpu.float(), lr=0.5, n_iters=3000
)

print("=== From-Scratch Gradient Descent ===")
print(f"Learned w: {w_scratch.tolist()}")
print(f"Learned b: {b_scratch.item():.4f}")

# Accuracy
z_scratch = X_cpu @ w_scratch + b_scratch
preds_scratch = (sigmoid(z_scratch) >= 0.5).float()
acc_scratch = (preds_scratch == y_cpu).float().mean().item()
print(f"Training accuracy: {acc_scratch:.4f}")

=== From-Scratch Gradient Descent ===
Learned w: [3.2906205654144287, 2.855818748474121]
Learned b: -0.2191
Training accuracy: 0.9800


## Validation Against scikit-learn

We compare:
1. **Accuracy** — both models should achieve near-identical training accuracy.
2. **Predicted probabilities** — using `C=1e6` (near-zero L2 penalty) for a fair apples-to-apples comparison.
3. **Coefficient direction** — cosine similarity should be > 0.999.

In [4]:
# sklearn reference with near-zero regularization
sk_model = LogisticRegression(C=1e6, solver="lbfgs", max_iter=5000, random_state=42)
sk_model.fit(X_cpu.numpy(), y_cpu.numpy().astype(int))

sk_w = torch.tensor(sk_model.coef_[0], dtype=torch.float32)
sk_b = torch.tensor(sk_model.intercept_, dtype=torch.float32)

y_sk_pred = sk_model.predict(X_cpu.numpy())
acc_sk = accuracy_score(y_cpu.numpy(), y_sk_pred)

print("=== scikit-learn Reference ===")
print(f"sklearn w:        {sk_w.tolist()}")
print(f"sklearn b:        {sk_b.item():.4f}")
print(f"sklearn accuracy: {acc_sk:.4f}")
print(f"From-scratch accuracy: {acc_scratch:.4f}")

# --- Assertion 1: Accuracy within 2% ---
acc_tol = 0.02
assert abs(acc_scratch - acc_sk) < acc_tol, (
    f"Accuracy gap too large: scratch={acc_scratch:.4f}, sklearn={acc_sk:.4f}"
)
print(f"\nAccuracy within {acc_tol:.0%}: PASS  (scratch={acc_scratch:.4f}, sklearn={acc_sk:.4f})")

# --- Assertion 2: Predicted probabilities are close ---
proba_scratch = sigmoid(X_cpu @ w_scratch + b_scratch.squeeze()).detach()
proba_sk = torch.tensor(sk_model.predict_proba(X_cpu.numpy())[:, 1], dtype=torch.float32)

prob_tol = 0.05
assert torch.allclose(proba_scratch, proba_sk, atol=prob_tol), (
    f"Probability mismatch (atol={prob_tol})  max diff: "
    f"{(proba_scratch - proba_sk).abs().max().item():.4f}"
)
print(f"Predicted probabilities within atol={prob_tol}: PASS")
print(f"  Max abs diff: {(proba_scratch - proba_sk).abs().max().item():.4f}")

=== scikit-learn Reference ===
sklearn w:        [3.296339988708496, 2.859269142150879]
sklearn b:        -0.2202
sklearn accuracy: 0.9800
From-scratch accuracy: 0.9800

Accuracy within 2%: PASS  (scratch=0.9800, sklearn=0.9800)
Predicted probabilities within atol=0.05: PASS
  Max abs diff: 0.0007


In [5]:
# --- Assertion 3: Coefficients point in the same direction ---
w_norm = w_scratch / w_scratch.norm()
sk_w_norm = sk_w / sk_w.norm()
cos_sim = (w_norm @ sk_w_norm).item()

cos_tol = 0.999
assert cos_sim > cos_tol, f"Coefficient direction mismatch: cos_sim={cos_sim:.6f}"
print(f"Coefficient cosine similarity: {cos_sim:.6f}  (> {cos_tol}): PASS")
print("\nAll assertions passed.")

Coefficient cosine similarity: 1.000000  (> 0.999): PASS

All assertions passed.


## Decision Boundary Plot

The decision boundary is the hyperplane where `z = xᵀw + b = 0`, i.e., `p(y=1|x) = 0.5`.

In [6]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

x_min = X_cpu[:, 0].min().item() - 0.5
x_max = X_cpu[:, 0].max().item() + 0.5
y_min = X_cpu[:, 1].min().item() - 0.5
y_max = X_cpu[:, 1].max().item() + 0.5
xx, yy = np.meshgrid(np.linspace(x_min, x_max, 300), np.linspace(y_min, y_max, 300))
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()], dtype=torch.float32)


def plot_boundary(ax, w, b_val, title):
    z_grid = (grid @ w + b_val).reshape(xx.shape).detach().numpy()
    proba_grid = 1 / (1 + np.exp(-z_grid))
    ax.contourf(xx, yy, proba_grid, levels=50, cmap="RdBu", alpha=0.6, vmin=0, vmax=1)
    ax.contour(xx, yy, z_grid, levels=[0], colors="black", linewidths=1.5)
    ax.scatter(
        X_cpu[y_cpu == 0, 0].numpy(), X_cpu[y_cpu == 0, 1].numpy(),
        c="blue", s=15, alpha=0.5, label="Class 0"
    )
    ax.scatter(
        X_cpu[y_cpu == 1, 0].numpy(), X_cpu[y_cpu == 1, 1].numpy(),
        c="red", s=15, alpha=0.5, label="Class 1"
    )
    ax.set_title(title)
    ax.legend(fontsize=8)
    ax.set_xlabel("Feature 1")
    ax.set_ylabel("Feature 2")


plot_boundary(axes[0], w_scratch, b_scratch.squeeze(), f"From Scratch (acc={acc_scratch:.3f})")
plot_boundary(axes[1], sk_w, sk_b.squeeze(), f"scikit-learn  (acc={acc_sk:.3f})")

plt.suptitle("Logistic Regression Decision Boundaries", fontsize=13)
plt.tight_layout()
plt.savefig("logistic_regression_boundary.png", dpi=100, bbox_inches="tight")
plt.show()
print("Plot saved.")

Plot saved.


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_71465/432743956.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
fig2, ax = plt.subplots(figsize=(7, 4))
ax.plot(loss_history, marker="o", markersize=4)
ax.set_xlabel("Iteration (x200)")
ax.set_ylabel("BCE Loss")
ax.set_title("Logistic Regression Training Loss (Gradient Descent)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("logistic_regression_loss.png", dpi=100, bbox_inches="tight")
plt.show()
print("Loss plot saved.")

Loss plot saved.


/var/folders/gx/cg22rrrs5mx_t0gwgx3809t80000gn/T/ipykernel_71465/1809592713.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Idiomatic Way: scikit-learn and PyTorch

For production use, prefer `sklearn.linear_model.LogisticRegression` for tabular data, or a `torch.nn.Linear` + `BCEWithLogitsLoss` for large-scale / GPU settings.

In [8]:
# scikit-learn idiomatic (default C=1.0 for L2 regularization)
sk_prod = LogisticRegression(solver="lbfgs", max_iter=1000, random_state=42)
sk_prod.fit(X_cpu.numpy(), y_cpu.numpy().astype(int))
print(f"sklearn (C=1.0) accuracy: {sk_prod.score(X_cpu.numpy(), y_cpu.numpy().astype(int)):.4f}")

# PyTorch idiomatic
import torch.nn as nn
import torch.optim as optim

torch.manual_seed(42)
model = nn.Linear(2, 1).to(device)
optimizer = optim.Adam(model.parameters(), lr=0.1)
loss_fn = nn.BCEWithLogitsLoss()

X_dev = X_cpu.to(device)
y_dev = y_cpu.unsqueeze(1).to(device)

for _ in range(2000):
    optimizer.zero_grad()
    logits = model(X_dev)
    loss = loss_fn(logits, y_dev)
    loss.backward()
    optimizer.step()

preds_nn = (torch.sigmoid(model(X_dev)) >= 0.5).float().cpu()
acc_nn = (preds_nn.squeeze() == y_cpu).float().mean().item()
print(f"PyTorch nn.Linear accuracy: {acc_nn:.4f}")
print(f"PyTorch w: {model.weight.data.cpu().squeeze().tolist()}")
print(f"PyTorch b: {model.bias.data.cpu().item():.4f}")

sklearn (C=1.0) accuracy: 0.9800


PyTorch nn.Linear accuracy: 0.9800
PyTorch w: [3.2947189807891846, 2.859692096710205]
PyTorch b: -0.2197


## Takeaways

- **Logistic regression** is a linear model for binary classification: it learns the log-odds as a linear function of features.
- **BCE loss** (binary cross-entropy) is the correct objective — it is the negative Bernoulli log-likelihood. See the `[[loss-functions]]` topic for derivation.
- **Gradients are elegant**: `nabla_w L = (1/n) X^T (p - y)` — identical in form to the linear regression gradient but with probabilities `p` instead of raw predictions.
- **Numerical stability matters**: never compute `log(sigmoid(z))` directly; use the log-sum-exp trick via `F.binary_cross_entropy_with_logits`.
- **Separability danger**: on perfectly separable data, coefficients grow to infinity. L2 regularization (`C < inf` in sklearn) keeps them finite.
- **Decision boundary** is the hyperplane `x^T w + b = 0`; shifting the probability threshold (e.g., from 0.5 to 0.3) moves the operating point on the precision-recall curve without retraining.
- Connect to `[[evaluation-metrics]]` for precision, recall, F1, ROC-AUC, and calibration diagnostics beyond accuracy.